# PROFUSEme: Predicting Prostate Cancer Recurrence with AI
### An educational walkthrough of a real medical AI research pipeline

---

## What is this notebook?

This notebook recreates the results of a real academic paper: **PROFUSEme** (*PROstate cancer recurrence prediction via FUSEd Multi-modal Embeddings*). The paper describes an AI system that can predict whether a prostate cancer patient's cancer will return after surgery — purely by analyzing their medical data.

**You don't need a background in AI or medicine to follow along.** Every step is explained in plain English before the code runs.

---

## The Medical Problem

After a patient has surgery to remove their prostate (radical prostatectomy), doctors monitor their **PSA** (Prostate-Specific Antigen) levels in the blood. PSA is a protein produced by prostate cells. If the cancer is gone, PSA should be undetectable.

If PSA levels rise again after surgery, it means cancer cells are likely still active somewhere in the body. This is called **Biochemical Recurrence (BCR)** — the cancer is coming back.

**The goal:** Given a patient's data at the time of surgery, can we predict who will experience BCR? If we can identify high-risk patients early, doctors can plan more aggressive follow-up treatment.

---

## The PROFUSEme Approach

Most AI systems for this problem only look at one type of data. PROFUSEme is special because it **combines three different data sources** — like a doctor consulting three different specialists:

```
┌─────────────────────────────────────────────────────────┐
│                     PATIENT DATA                        │
│                                                         │
│  📋 Clinical Chart  +  🔬 Tissue Slide  +  🧲 MRI Scan │
│    (8 numbers)         (thousands of        (3D scan)   │
│                          patches)                       │
└─────────────┬───────────────┬───────────────┬───────────┘
              │               │               │
              ▼               ▼               ▼
         Encode into     Extract patch    Extract MRI
         numbers         features         features
              │               │               │
              └───────────────┴───────────────┘
                              │
                              ▼
                    FUSE (combine) all data
                              │
                              ▼
                   PREDICT: BCR or No BCR?
```

---

## How to use this notebook

Run each cell from top to bottom by clicking on it and pressing **Shift + Enter** (or clicking the ▶ Run button).

**Before you start:**
- Make sure `client_secrets.json` is in the same folder as this notebook (for Google Drive upload)
- Internet connection required (data streams from a public research database)

> **Note:** Section 8 (Google Drive upload) will open a browser tab. Click **Allow** when prompted.

---
## Section 0 — Setup
### Installing and importing everything we need

In [ ]:
# Install dependencies if not already installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "boto3", "s3fs", "torch", "scikit-learn",
    "matplotlib", "seaborn", "pandas", "numpy", "scipy",
    "pydrive2", "google-auth", "PyYAML", "ipywidgets"
])
print("✅ All packages ready")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Import our backend module
from profuseme_utils import (
    S3Client, ClinicalLoader, PathologyLoader, RadiologyLoader,
    MultiModalFusion, BCRPredictor, align_patients, PATIENTS
)

# Plotting style
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_theme(style="whitegrid", palette="Set2")

print("✅ Imports complete")
print(f"   Working with {len(PATIENTS)} patients: {PATIENTS[:5]}...")

---
## Section 1 — What Data Do We Have?

### The CHIMERA Dataset

This project uses the **CHIMERA** dataset — a public research dataset of prostate cancer patients. Think of it like a shared library of anonymized patient records that researchers around the world can access freely.

The data lives in **Amazon S3**, a cloud storage service (like Dropbox but for researchers). We'll stream the data directly from there — no need to download gigabytes of files to your computer.

Each patient in CHIMERA has three types of data:

| Data Type | What it is | Size |
|-----------|-----------|------|
| **Clinical chart** | Medical history in JSON format (a structured text file) | ~5 KB |
| **Pathology slide** | A digitized microscope slide of the prostate tissue | ~3 GB |
| **MRI scan** | 3D magnetic resonance images of the prostate | ~500 MB |

Because the slides and MRI are enormous, we won't download them. Instead, we use **pre-computed features** — compact numerical summaries that researchers have already extracted from these images.

In [ ]:
# Connect to the CHIMERA research database (S3)
print("Connecting to the CHIMERA research database...")
s3 = S3Client()
print(f"\n✅ Connected! We have {len(PATIENTS)} patients available.")
print("\nPatient IDs:")
print("  ", ", ".join(PATIENTS))

In [ ]:
# Peek at one patient's clinical record to understand the data format
import json

clinical_loader = ClinicalLoader()
sample_record = clinical_loader.load_patient(PATIENTS[0], s3)

print(f"Clinical record for patient {PATIENTS[0]}:")
print("─" * 50)
for key, value in sample_record.items():
    print(f"  {key:40s}: {value}")

---
## Section 2 — Clinical Data: The Patient's Chart

### What is clinical data?

Clinical data is the information recorded in a patient's medical chart — things a doctor writes down during an appointment or after surgery. For prostate cancer, the most important clinical factors are:

| Feature | Plain English explanation |
|---------|---------------------------|
| **Age** | Older patients may have different risk profiles |
| **ISUP Grade** | A score (1–5) describing how abnormal the cancer cells look under a microscope. Higher = more aggressive |
| **pT Stage** | How far the cancer had spread at surgery (pT2 = confined to prostate, pT4 = spread to nearby organs) |
| **Lymph Node Involvement** | Were cancer cells found in nearby lymph nodes? (Yes/No) |
| **Surgical Margins** | Did the surgeon get all the cancer cells? Positive margins = some may remain |
| **Seminal Vesicle Invasion** | Did the cancer invade the seminal vesicles? (Yes/No) |
| **Lymphovascular Invasion** | Did cancer cells get into blood or lymph vessels? (Yes/No) |
| **Capsular Penetration** | Did the cancer break through the prostate capsule (outer wall)? (Yes/No) |

### Turning medical facts into numbers

Computers work with numbers, not text. So we convert the clinical record into a vector of 8 numbers — one for each feature above. This is called **feature encoding**.

In [ ]:
# Load clinical data for ALL patients
print("Loading clinical data for all patients...")
X_clin, y, clinical_ids = clinical_loader.load_all(PATIENTS, s3)
feature_names = clinical_loader.get_feature_names()

print(f"\n✅ Loaded {len(clinical_ids)} patients")
print(f"   Clinical feature matrix shape: {X_clin.shape}  (patients × features)")
print(f"   BCR labels: {y.tolist()}")
print(f"   BCR-positive patients: {y.sum()} / {len(y)}")

In [ ]:
# Visualize the clinical dataset
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: BCR distribution
ax = axes[0]
counts = [int((y == 0).sum()), int((y == 1).sum())]
bars = ax.bar(["No Recurrence (BCR=0)", "Recurrence (BCR=1)"], counts,
              color=["#4CAF50", "#F44336"], edgecolor="white", linewidth=1.5)
ax.set_title("Patient Outcomes", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Patients")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(count), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(counts) + 2)

# Right: Feature heatmap
ax = axes[1]
df_feat = pd.DataFrame(X_clin, columns=feature_names, index=clinical_ids)
df_feat["BCR"] = y
df_feat = df_feat.sort_values("BCR")
sns.heatmap(
    df_feat.drop(columns="BCR").T,
    ax=ax, cmap="RdBu_r", center=0,
    xticklabels=df_feat.index, yticklabels=feature_names,
    cbar_kws={"label": "Feature value"}
)
ax.set_title("Clinical Features per Patient", fontsize=13, fontweight="bold")
ax.set_xlabel("Patient ID (sorted by BCR)")
# Add BCR label row indicator
for i, pid in enumerate(df_feat.index):
    bcr = df_feat.loc[pid, "BCR"]
    ax.axvline(i + 0.5, color="red" if bcr else "green", linewidth=1.5, alpha=0.4)

plt.tight_layout()
plt.suptitle("Section 2: Clinical Data Overview", y=1.02, fontsize=14, fontweight="bold")
plt.savefig("results_clinical.png", dpi=150, bbox_inches="tight")
plt.show()
print("Red columns = BCR-positive patients, Green = BCR-negative")

---
## Section 3 — Pathology Data: The Tissue Slides

### What is a whole-slide image (WSI)?

After surgery, the removed prostate is sliced very thin and stained with dyes so the cells show up under a microscope. This slice is then digitally scanned at extremely high resolution — creating a **Whole Slide Image (WSI)**. A single WSI can be **3+ gigabytes** and contain billions of pixels.

Pathologists look at these slides to identify cancer cells and grade how abnormal they look (this is where ISUP grade comes from).

### THINKING FAST vs THINKING SLOW

The PROFUSEme paper uses a two-scale approach inspired by how humans think:

- **THINKING FAST** (224×224 pixel patches at 1.25× magnification): Like stepping back to see the whole picture — get a broad overview of the tissue architecture
- **THINKING SLOW** (1024×1024 pixel patches at 20× magnification): Like zooming in close — see the fine details of individual cells

### Feature extraction: Describing slides with numbers

Instead of sending billions of pixels to our model, we use a **pre-trained neural network** to compress each patch into a compact description — a vector of ~512 numbers that capture the important visual patterns. This is called a **feature vector** or **embedding**.

The CHIMERA dataset provides these pre-computed embeddings, so we don't need to run the neural network ourselves.

In [ ]:
# Load pre-computed pathology features from S3
print("Loading pathology features from the CHIMERA database...")
print("(This streams compact feature files, not the giant WSI images)")

path_loader = PathologyLoader(s3)
X_path, path_ids = path_loader.load_all(clinical_ids)

print(f"\n✅ Loaded pathology features for {len(path_ids)} patients")
print(f"   Feature matrix shape: {X_path.shape}")
print(f"   Each patient is described by {X_path.shape[1]} numbers")

In [ ]:
# Visualize patch coordinates (the grid of patches extracted from a slide)
viz_patient = path_ids[0] if path_ids else clinical_ids[0]
coords = path_loader.load_coordinates(viz_patient, slide_num=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Patch grid visualization
ax = axes[0]
if coords is not None:
    ax.scatter(coords['x'], coords['y'], s=2, alpha=0.5, color="steelblue")
    ax.invert_yaxis()
    ax.set_title(f"Patch Grid: Patient {viz_patient}", fontweight="bold")
    ax.set_xlabel("X coordinate (pixels)")
    ax.set_ylabel("Y coordinate (pixels)")
    ax.text(0.05, 0.95, f"{len(coords):,} patches extracted",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8))
else:
    ax.text(0.5, 0.5, "Coordinate data not available",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Patch Grid (unavailable)")

# Right: Feature vector visualization
ax = axes[1]
feat_sample = X_path[0]
ax.bar(range(len(feat_sample)), feat_sample, color="steelblue", alpha=0.7, width=1.0)
ax.set_title(f"Pathology Feature Vector: Patient {viz_patient}", fontweight="bold")
ax.set_xlabel(f"Feature dimension (0 – {len(feat_sample)-1})")
ax.set_ylabel("Feature value")
ax.text(0.05, 0.95,
    f"Compressed tissue slide description\n— {len(feat_sample)} numbers",
    transform=ax.transAxes, fontsize=10,
    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

plt.suptitle("Section 3: Pathology Data", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results_pathology.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 4 — Radiology Data: The MRI Scans

### What is prostate MRI?

MRI (Magnetic Resonance Imaging) uses magnetic fields and radio waves to create detailed 3D images of soft tissues inside the body without radiation. For prostate cancer, doctors typically use three complementary types of MRI sequences:

| Sequence | What it shows |
|----------|---------------|
| **T2W** (T2-Weighted) | Anatomy — the shape and structure of the prostate |
| **ADC** (Apparent Diffusion Coefficient) | How freely water molecules can move — cancer restricts movement |
| **HBV** (High b-Value DWI) | Highlights areas where water movement is very restricted |

Together, these three views give a complete picture that a radiologist uses to identify and assess tumors.

### Pre-computed MRI features

Just like the tissue slides, the CHIMERA dataset provides pre-computed feature vectors for the MRI scans. A specialized neural network has already processed the 3D scans and compressed each one into a compact vector of numbers.

In [ ]:
# Load radiology features from S3
print("Loading MRI features from the CHIMERA database...")

rad_loader = RadiologyLoader(s3)
X_rad, rad_ids = rad_loader.load_all(clinical_ids)

print(f"\n✅ Loaded radiology features for {len(rad_ids)} patients")
print(f"   Feature matrix shape: {X_rad.shape}")
print(f"   Each patient's MRI is described by {X_rad.shape[1]} numbers")

In [ ]:
# Visualize radiology features across patients
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Feature vector for one patient
ax = axes[0]
ax.bar(range(X_rad.shape[1]), X_rad[0], color="coral", alpha=0.7, width=1.0)
ax.set_title(f"MRI Feature Vector: Patient {rad_ids[0]}", fontweight="bold")
ax.set_xlabel("Feature dimension")
ax.set_ylabel("Feature value")
ax.text(0.05, 0.95,
    f"The 3D MRI scan compressed\ninto {X_rad.shape[1]} numbers",
    transform=ax.transAxes, fontsize=10,
    bbox=dict(boxstyle="round", facecolor="#FFE0D9", alpha=0.9))

# Right: Compare average feature magnitude across BCR groups
ax = axes[1]
# Align radiology with labels
aligned = align_patients(clinical_ids, y, X_clin,
                         rad_ids=rad_ids, X_rad=X_rad)
y_aligned = aligned["y"]
X_rad_aligned = aligned["X_rad"]

if X_rad_aligned is not None:
    mean_bcr_pos = np.abs(X_rad_aligned[y_aligned == 1]).mean(axis=0)
    mean_bcr_neg = np.abs(X_rad_aligned[y_aligned == 0]).mean(axis=0)
    # Show first 64 dims for readability
    dims = min(64, X_rad.shape[1])
    ax.plot(mean_bcr_neg[:dims], label="BCR-negative", color="#4CAF50", alpha=0.8)
    ax.plot(mean_bcr_pos[:dims], label="BCR-positive", color="#F44336", alpha=0.8)
    ax.set_title("MRI Feature Magnitude by BCR Status", fontweight="bold")
    ax.set_xlabel("Feature dimension (first 64)")
    ax.set_ylabel("|Feature value|")
    ax.legend()
    ax.text(0.05, 0.95,
        "Differences between groups\nsuggest MRI carries BCR signal",
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

plt.suptitle("Section 4: Radiology (MRI) Data", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results_radiology.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 5 — Multi-Modal Fusion: Combining Everything

### Why combine data sources?

Imagine you're trying to decide if a fruit is ripe. You might:
- **Look** at its color (visual data)
- **Smell** it (chemical data)  
- **Squeeze** it to feel the firmness (mechanical data)

Any one of these alone might mislead you. Combined, they're much more reliable. The same is true for cancer prediction.

### The dimensionality problem

Our three data sources produce vectors of very different sizes:
- Clinical: **8 numbers**
- Pathology: **512 numbers** (or more)
- Radiology: **512 numbers** (or more)

If we just concatenated them, we'd have 1032+ numbers per patient — but only 15 patients! A model with more inputs than examples will just memorize random patterns (overfitting).

### PCA: Finding the most important patterns

**Principal Component Analysis (PCA)** is a mathematical technique that finds the directions of maximum variation in a dataset. It's like finding the "main themes" in a piece of music — instead of describing every note, you describe the key patterns.

We use PCA to compress each modality from 512 dimensions down to 8 — keeping the most informative patterns while discarding noise. Then we concatenate the compressed versions:

```
Clinical (8)  +  Pathology-PCA (8)  +  Radiology-PCA (8)  =  24 numbers per patient
```

24 features for 15 patients is manageable for a simple model.

In [ ]:
# Align all three modalities to the same patients
aligned = align_patients(
    clinical_ids, y, X_clin,
    path_ids=path_ids, X_path=X_path,
    rad_ids=rad_ids, X_rad=X_rad
)

patients_final = aligned["patients"]
y_final        = aligned["y"]
X_clin_final   = aligned["X_clin"]
X_path_final   = aligned["X_path"]
X_rad_final    = aligned["X_rad"]

print(f"Patients with all three data types: {len(patients_final)}")
print(f"BCR-positive: {y_final.sum()}  |  BCR-negative: {(y_final == 0).sum()}")

In [ ]:
# Visualize fusion with a 2D PCA scatter plot
# (We use PCA on ALL data here just for visualization — the model uses it correctly per-fold)

fusion = MultiModalFusion(path_components=8, rad_components=8)
train_mask_all = np.ones(len(y_final), dtype=bool)
X_fused_viz = fusion.fuse(X_clin_final, X_path_final, X_rad_final, train_mask_all)

# 2D PCA of the fused representation
pca2d = PCA(n_components=2)
scaler = StandardScaler()
X_2d = pca2d.fit_transform(scaler.fit_transform(X_fused_viz))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 2D scatter of patients
ax = axes[0]
ax.scatter(X_2d[y_final == 0, 0], X_2d[y_final == 0, 1],
           c="#4CAF50", s=120, label="No Recurrence", edgecolors="black", linewidths=0.5)
ax.scatter(X_2d[y_final == 1, 0], X_2d[y_final == 1, 1],
           c="#F44336", s=120, marker="*", label="Recurrence (BCR)", edgecolors="black", linewidths=0.5)
for i, pid in enumerate(patients_final):
    ax.annotate(pid, (X_2d[i, 0], X_2d[i, 1]),
                fontsize=7, ha="center", va="bottom", color="gray")
ax.set_xlabel(f"PCA Component 1 ({pca2d.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PCA Component 2 ({pca2d.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("All 15 Patients in Fused Feature Space", fontweight="bold")
ax.legend()
ax.text(0.05, 0.05,
    "If BCR+ and BCR- cluster\nseparately → model should work!",
    transform=ax.transAxes, fontsize=9,
    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

# Right: Explain PCA with a simple analogy
ax = axes[1]
ax.axis("off")
explanation = (
    "PCA in Plain English:\n"
    "──────────────────────\n"
    "Imagine plotting all 15 patients\n"
    "in 512-dimensional space (impossible\n"
    "to visualize).\n\n"
    "PCA finds the 2 directions where\n"
    "patients differ the MOST, and\n"
    "projects everyone onto those.\n\n"
    "It's like flattening a 3D cloud\n"
    "of points onto the 2D plane that\n"
    "shows the most spread-out view.\n\n"
    "For the model, we keep 8 components\n"
    "per modality instead of 2, to retain\n"
    "more information."
)
ax.text(0.5, 0.5, explanation, ha="center", va="center", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="#E3F2FD", alpha=0.95),
        transform=ax.transAxes, multialignment="left")

plt.suptitle("Section 5: Multi-Modal Fusion", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("results_fusion.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 6 — Prediction Model: Can We Predict Recurrence?

### What model do we use?

We use **Logistic Regression** — one of the simplest and most interpretable machine learning models. Despite its simplicity, it's surprisingly powerful and perfectly appropriate when:
- You have very few samples (we have only 15 patients)
- You need to avoid overfitting
- Interpretability matters (doctors need to trust and understand the model)

Think of logistic regression as: **"each feature gets a vote, weighted by how much we trust it, and the total vote determines BCR or no BCR."**

### Leave-One-Out Cross-Validation (LOO-CV)

With only 15 patients, we can't afford to set aside a large test set. Instead we use **LOO-CV**:

1. Train on 14 patients → Predict on the 1 left-out patient
2. Repeat 15 times (each patient gets left out once)
3. Collect all 15 predictions and compute metrics

This is the most honest way to evaluate a model on a tiny dataset.

### How to measure success: AUC

**AUC** (Area Under the ROC Curve) measures how well the model *ranks* patients by risk:
- **AUC = 1.0**: Perfect — BCR patients always get higher risk scores than non-BCR patients
- **AUC = 0.5**: Random — no better than flipping a coin
- **AUC = 0.7–0.8**: Clinically useful — the model has real predictive power

In [ ]:
# Run the prediction model with leave-one-out cross-validation
print("Running leave-one-out cross-validation...")
print("(Training 15 separate models, each leaving one patient out)")
print()

predictor = BCRPredictor(C=0.01)
loo_result = predictor.evaluate_loo(
    X_clin_final, y_final,
    X_path=X_path_final, X_rad=X_rad_final
)

if "error" in loo_result:
    print(f"⚠️  Could not compute AUC: {loo_result['error']}")
else:
    print("═" * 50)
    print("  MODEL RESULTS (Leave-One-Out CV)")
    print("═" * 50)
    print(f"  AUC:          {loo_result['auc']:.3f}  (95% CI: {loo_result['ci_low']:.3f} – {loo_result['ci_high']:.3f})")
    print(f"  Accuracy:     {loo_result['accuracy']:.3f}")
    print(f"  Sensitivity:  {loo_result['sensitivity']:.3f}  (how often we catch BCR patients)")
    print(f"  Specificity:  {loo_result['specificity']:.3f}  (how often we correctly say 'no BCR')")
    print("═" * 50)
    print()
    if loo_result['auc'] >= 0.75:
        print("  ✅ Good predictive performance!")
    elif loo_result['auc'] >= 0.60:
        print("  ⚠️  Moderate performance — reasonable for a 15-patient dataset")
    else:
        print("  ℹ️  Low AUC — may reflect the very small sample size (15 patients)")

In [ ]:
# Visualize model results
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

proba = np.array(loo_result["predicted_proba"])
true_labels = np.array(loo_result["true_labels"])

# Left: ROC curve
ax = axes[0]
fpr, tpr, _ = roc_curve(true_labels, proba)
auc_val = loo_result["auc"]
ax.plot(fpr, tpr, color="#1565C0", linewidth=2.5,
        label=f"PROFUSEme (AUC = {auc_val:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1.5, label="Random guess (AUC = 0.50)")
ax.fill_between(fpr, tpr, alpha=0.1, color="#1565C0")
ax.set_xlabel("False Positive Rate\n(% of healthy patients incorrectly flagged)")
ax.set_ylabel("True Positive Rate\n(% of BCR patients correctly identified)")
ax.set_title("ROC Curve", fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.text(0.05, 0.95,
    "A curve hugging the top-left\ncorner = good model",
    transform=ax.transAxes, fontsize=9, va="top",
    bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.9))

# Right: Predicted probability per patient
ax = axes[1]
sorted_idx = np.argsort(proba)
sorted_proba = proba[sorted_idx]
sorted_labels = true_labels[sorted_idx]
sorted_pids = [patients_final[i] for i in sorted_idx]
bar_colors = ["#F44336" if lb else "#4CAF50" for lb in sorted_labels]
bars = ax.barh(range(len(sorted_pids)), sorted_proba, color=bar_colors, edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="Decision threshold (0.5)")
ax.set_yticks(range(len(sorted_pids)))
ax.set_yticklabels(sorted_pids, fontsize=8)
ax.set_xlabel("Predicted Probability of Recurrence")
ax.set_title("Risk Score per Patient", fontweight="bold")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
green_patch = mpatches.Patch(color="#4CAF50", label="BCR-negative (no recurrence)")
red_patch   = mpatches.Patch(color="#F44336", label="BCR-positive (recurrence)")
ax.legend(handles=[green_patch, red_patch], loc="lower right", fontsize=8)

plt.suptitle("Section 6: Prediction Results", fontsize=14, fontweight="bold")
plt.tight_layout()

# Save the figure for Drive upload
roc_fig = fig
plt.savefig("results_roc.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Section 7 — Ablation Study: Which Data Source Matters Most?

### What is an ablation study?

In neuroscience, an "ablation" means removing part of the brain to see what function is lost. In AI research, we borrow this term to mean: **remove one ingredient and see how much worse the recipe gets.**

We test all 7 combinations of our three data sources:

| Combination | What it uses |
|-------------|-------------|
| Clinical only | Just the 8 numbers from the patient chart |
| Pathology only | Just the tissue slide features |
| Radiology only | Just the MRI features |
| Clinical + Pathology | No MRI |
| Clinical + Radiology | No tissue slide |
| Pathology + Radiology | No chart data |
| **All three** | **The full PROFUSEme approach** |

This tells us which sources contribute most to prediction accuracy, and whether combining all three really helps.

In [ ]:
# Run the ablation study
print("Running ablation study (7 modality combinations × 15 LOO folds = 105 model fits)...")
print("This may take a minute.")
print()

ablation_df = predictor.ablation_study(
    X_clin_final, y_final,
    X_path=X_path_final, X_rad=X_rad_final
)

print("Results:")
print(ablation_df.to_string(index=False))

In [ ]:
# Visualize the ablation study
fig, ax = plt.subplots(figsize=(9, 5))

auc_values = ablation_df["AUC"].values
combos = ablation_df["Modality Combination"].values
bar_colors = ["#1565C0" if "All three" in c else "#64B5F6" for c in combos]

bars = ax.barh(range(len(combos)), auc_values, color=bar_colors, edgecolor="white", height=0.6)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1.5, alpha=0.7, label="Random (AUC=0.5)")

# Label bars with AUC values
for i, (bar, val) in enumerate(zip(bars, auc_values)):
    if not np.isnan(val):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9, fontweight="bold")

ax.set_yticks(range(len(combos)))
ax.set_yticklabels(combos, fontsize=10)
ax.set_xlabel("AUC (Area Under ROC Curve)", fontsize=11)
ax.set_title("Ablation Study: AUC by Modality Combination", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.05)

dark_patch = mpatches.Patch(color="#1565C0", label="Full PROFUSEme model")
light_patch = mpatches.Patch(color="#64B5F6", label="Single/partial modality")
ax.legend(handles=[dark_patch, light_patch], loc="lower right")

plt.tight_layout()
plt.savefig("results_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print()
print("Interpretation: The higher the AUC, the better the model.")
print("If the 'All three' bar is highest, multi-modal fusion truly helps!")

In [29]:
from pathlib import Path

if not Path("client_secrets.json").exists():
    print("⚠️  client_secrets.json not found in this directory.")
    print("   Please follow the instructions in GDRIVE_SETUP.md")
    print("   and place the file here before running this cell.")
else:
    print("✅ client_secrets.json found — connecting to Google Drive...")
    print("   A browser tab will open. Log in and click Allow.")
    print()

    uploader = DriveUploader()

    uploader.upload_results(
        loo_result=loo_result,
        ablation_df=ablation_df,
        roc_fig=roc_fig,
        X_clin=X_clin_final,
        X_path=X_path_final,
        X_rad=X_rad_final,
        patient_ids=list(patients_final),
    )

    print()
    print("🎉 Done! Check Google Drive for a folder called 'PROFUSEme_Data'.")

✅ client_secrets.json found — connecting to Google Drive...
   A browser tab will open. Log in and click Allow.

A browser tab will open — log in and click Allow.
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=32912672338-gbfjvnj55s17k3tq1hbbr0b3vetiekop.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code



INFO: Received token response with no refresh_token. Consider reauthenticating with prompt='consent'.
INFO: Successfully retrieved access token


Authentication successful.


INFO: Drive ready: PROFUSEme_Data/Results/


ApiRequestError: <HttpError 403 when requesting https://www.googleapis.com/upload/drive/v2/files?supportsAllDrives=true&alt=json&uploadType=resumable returned "The user has exceeded their Drive storage quota". Details: "[{'message': 'The user has exceeded their Drive storage quota', 'domain': 'usageLimits', 'reason': 'quotaExceeded'}]">

import json
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# loo_predictions.csv
preds_df = pd.DataFrame({
    "patient_id": list(patients_final),
    "true_bcr": loo_result["true_labels"],
    "predicted_proba": loo_result["predicted_proba"],
})
preds_df.to_csv(results_dir / "loo_predictions.csv", index=False)

# ablation_table.csv
ablation_df.to_csv(results_dir / "ablation_table.csv", index=False)

# summary.json
summary = {
    "auc": round(loo_result["auc"], 4),
    "ci_95": [round(loo_result["ci_low"], 4), round(loo_result["ci_high"], 4)],
    "accuracy": round(loo_result["accuracy"], 4),
    "sensitivity": round(loo_result["sensitivity"], 4),
    "specificity": round(loo_result["specificity"], 4),
    "n_patients": len(loo_result["true_labels"]),
    "n_bcr_positive": int(sum(loo_result["true_labels"])),
}
with open(results_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Copy result PNGs into results/
import shutil
for png in Path(".").glob("results_*.png"):
    shutil.copy(png, results_dir / png.name)

print("✅ All results saved to ./results/")
for p in sorted(results_dir.iterdir()):
    print(f"   {p.name}")